In [ ]:
import pandas as pd
import numpy as np
import statistics

ECOTOX data

In [ ]:
ecotox = pd.read_csv('../../Final_tables/Output/FINAL_DATA/MAIN_ecotox_toxicity_data.tsv', sep='\t', dtype=str)
ecotox = ecotox.fillna('')
print(ecotox.shape)
ecotox.head()

In [ ]:
ecotox['ecotox_species_group'].value_counts()

In [ ]:
ecotox['endpoint'].value_counts()

In [ ]:
ecotox['casrn'] = ecotox['casrn'].str.replace('CAS:','')
print(ecotox.shape)
ecotox.head()

In [ ]:
ecotox['ppm_equivalent'] = ecotox['ppm_equivalent'].astype(float)

In [ ]:
chronic_endp = ['NOEC', 'LOEL', 'NOEL']
acute_endp = ['EC50', 'LC50']

Food web data

In [ ]:
toxdata = pd.read_csv('../../Final_tables/Output/FINAL_DATA/MAIN_foodweb_feed_tox.tsv', sep='\t', dtype=str)
toxdata['casrn'] = toxdata['casrn'].str.replace('CAS:','')
toxdata = toxdata.fillna('')
print(toxdata.shape)
toxdata.head()

## Acute toxicity

In [ ]:
toxdata_acute = toxdata[toxdata['toxicity_type']=='Acute']
toxdata_acute = toxdata_acute.drop(columns=['aquaculture_predator_species']).drop_duplicates()
print(toxdata_acute.shape)
toxdata_acute.head()

In [ ]:
acute_sp = set(toxdata_acute['prey_species']) - {''}
print(len(acute_sp))

In [ ]:
len(set(ecotox[ecotox['latin_name'].isin(acute_sp)]['ecotox_species_group']))

In [ ]:
acute_chemgroup = {'Chemical': [], 'Chemical_name': [], 'Species': []}

for k in set(ecotox['id']):
    tdf = ecotox[(ecotox['id'] == k) & (ecotox['endpoint'].isin(acute_endp))]
    if len(set(tdf['latin_name'])) >= 8:
        cas = list(set(tdf['casrn']))[0]
        cname = list(set(tdf['chemical_name']))[0]
        sps = len(set(tdf['latin_name']))
        acute_chemgroup['Chemical'].append(cas)
        acute_chemgroup['Chemical_name'].append(cname)
        acute_chemgroup['Species'].append(sps)

print(len(set(ecotox['casrn'])))
print(len(acute_chemgroup['Chemical']))

In [ ]:
acute_ssd_eligible = pd.DataFrame(acute_chemgroup)
acute_ssd_eligible['Type'] = 'Acute'
print(acute_ssd_eligible.shape)
acute_ssd_eligible.head()

In [ ]:
ecotox.columns

In [ ]:
acute_df = ecotox[(ecotox['casrn'].isin(acute_chemgroup['Chemical']))&(ecotox['endpoint'].isin(acute_endp))]
acute_df = acute_df[['casrn', 'chemical_name', 'latin_name',
       'ecotox_species_group',
       'ppm_equivalent']]
print(acute_df.shape)
acute_df.head()

In [ ]:
acute_df = acute_df.groupby(['casrn', 'chemical_name', 'latin_name',
       'ecotox_species_group']).agg({'ppm_equivalent': lambda x:statistics.geometric_mean([i for i in list(x)])}).reset_index()
print(acute_df.shape)
acute_df.head()

In [ ]:
acute_df = acute_df[['casrn', 'chemical_name', 'latin_name', 'ecotox_species_group', 'ppm_equivalent']]
acute_df.columns = ['CAS', 'Chemical', 'Latin Name', 'Group', 'Conc']
print(acute_df.shape)
acute_df.head()

In [ ]:
acute_df.to_csv('./ssd_input/ssd_input_acute_data.tsv', sep='\t', index=False, encoding='utf-8')

In [ ]:
for k in set(acute_df['CAS']):
    tmp = acute_df[acute_df['CAS']==k]
    tmp.to_csv(f'./ssd_input/acute/{k}.csv', index=False, encoding='utf-8')

## Chronic toxicity

In [ ]:
toxdata_chronic = toxdata[toxdata['toxicity_type']=='Chronic']
toxdata_chronic = toxdata_chronic.drop(columns=['aquaculture_predator_species']).drop_duplicates()
print(toxdata_chronic.shape)
toxdata_chronic.head()

In [ ]:
chronic_sp = set(toxdata_chronic['prey_species']) - {''}
print(len(chronic_sp))

In [ ]:
len(set(ecotox[ecotox['latin_name'].isin(chronic_sp)]['ecotox_species_group']))

In [ ]:
chronic_chemgroup = {'Chemical': [], 'Chemical_name': [], 'Species': []}

for k in set(ecotox['id']):
    tdf = ecotox[(ecotox['id'] == k) & (ecotox['endpoint'].isin(chronic_endp))]
    if len(set(tdf['latin_name'])) >= 8:
        cas = list(set(tdf['casrn']))[0]
        cname = list(set(tdf['chemical_name']))[0]
        sps = len(set(tdf['latin_name']))
        chronic_chemgroup['Chemical'].append(cas)
        chronic_chemgroup['Chemical_name'].append(cname)
        chronic_chemgroup['Species'].append(sps)

print(len(set(ecotox['casrn'])))
print(len(chronic_chemgroup['Chemical']))

In [ ]:
chronic_ssd_eligible = pd.DataFrame(chronic_chemgroup)
chronic_ssd_eligible['Type'] = 'chronic'
print(chronic_ssd_eligible.shape)
chronic_ssd_eligible.head()

In [ ]:
chronic_df = ecotox[(ecotox['casrn'].isin(acute_chemgroup['Chemical']))&(ecotox['endpoint'].isin(chronic_endp))]
chronic_df = chronic_df[['casrn', 'chemical_name', 'latin_name',
       'ecotox_species_group',
       'ppm_equivalent']]
print(chronic_df.shape)
chronic_df.head()

In [ ]:
chronic_df = chronic_df.groupby(['casrn', 'chemical_name', 'latin_name',
       'ecotox_species_group']).agg({'ppm_equivalent': lambda x:statistics.geometric_mean([i for i in list(x)])}).reset_index()
print(chronic_df.shape)
chronic_df.head()

In [ ]:
chronic_df = chronic_df[['casrn', 'chemical_name', 'latin_name', 'ecotox_species_group', 'ppm_equivalent']]
chronic_df.columns = ['CAS', 'Chemical', 'Latin Name', 'Group', 'Conc']
print(chronic_df.shape)
chronic_df.head()

In [ ]:
chronic_df.to_csv('./ssd_input/ssd_input_chronic_data.tsv', sep='\t', index=False, encoding='utf-8')

In [ ]:
for k in set(chronic_df['CAS']):
    tmp = chronic_df[chronic_df['CAS']==k]
    tmp.to_csv(f'./ssd_input/chronic/{k}.csv', index=False, encoding='utf-8')

# Combine data

In [ ]:
ssd_chem = pd.concat([acute_ssd_eligible, chronic_ssd_eligible])
print(ssd_chem.shape)
ssd_chem.head()

In [ ]:
ssd_chem.to_csv('./ssd_input/ssd_chemicals.tsv', sep='\t', index=False, encoding='utf-8')